# 🧮 Model v1 — Elo Ratings
**World Cup 2026 Predictor** · Computing Elo ratings from scratch over 96 years of World Cup history, and testing whether they beat the v0 features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

PATH = "../data/raw"
matches = pd.read_csv(PATH + "/historical/matches.csv")
fixture = pd.read_csv(PATH + "/fixture_2026/fixture_2026.csv")

# Same prep as notebook 02
matches_m = matches[matches["tournament_name"].str.contains("Men's")].copy()
matches_m["year"] = matches_m["tournament_name"].str[:4].astype(int)

HEIR_NATIONS = {
    "West Germany": "Germany", "Soviet Union": "Russia", "Yugoslavia": "Serbia",
    "Serbia and Montenegro": "Serbia", "Czechoslovakia": "Czech Republic",
    "Zaire": "DR Congo", "Dutch East Indies": "Indonesia",
}
matches_m["home_team_name"] = matches_m["home_team_name"].replace(HEIR_NATIONS)
matches_m["away_team_name"] = matches_m["away_team_name"].replace(HEIR_NATIONS)

fixture["team1"] = fixture["team1"].replace({"USA": "United States", "Bosnia & Herzegovina": "Bosnia and Herzegovina"})
fixture["team2"] = fixture["team2"].replace({"USA": "United States", "Bosnia & Herzegovina": "Bosnia and Herzegovina"})

# Sort chronologically — CRITICAL for Elo (ratings evolve match by match)
matches_m["match_date"] = pd.to_datetime(matches_m["match_date"])
matches_m = matches_m.sort_values("match_date").reset_index(drop=True)
print(len(matches_m), "matches from", matches_m["match_date"].min().date(), "to", matches_m["match_date"].max().date())

## 1. Computing Elo ratings match by match

In [2]:
def expected_score(rating_a, rating_b):
    """Expected probability that A beats B given their ratings (classic Elo formula)."""
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

K = 40           # update speed: how much one match can move a rating
BASE_ELO = 1500  # every team starts here

elo = {}                  # current rating of each team
elo_home_list = []        # rating of home team BEFORE each match (this is the feature!)
elo_away_list = []

for _, m in matches_m.iterrows():
    home, away = m["home_team_name"], m["away_team_name"]
    r_home = elo.get(home, BASE_ELO)
    r_away = elo.get(away, BASE_ELO)

    # Store PRE-match ratings → features without leakage by construction
    elo_home_list.append(r_home)
    elo_away_list.append(r_away)

    # Actual result: 1 = home win, 0.5 = draw, 0 = away win
    if m["home_team_win"] == 1:
        score = 1.0
    elif m["draw"] == 1:
        score = 0.5
    else:
        score = 0.0

    # Update ratings
    exp = expected_score(r_home, r_away)
    elo[home] = r_home + K * (score - exp)
    elo[away] = r_away + K * ((1 - score) - (1 - exp))

matches_m["elo_home"] = elo_home_list
matches_m["elo_away"] = elo_away_list
matches_m["elo_diff"] = matches_m["elo_home"] - matches_m["elo_away"]

# Sanity check: current top 10 (after processing all of history)
ranking = pd.Series(elo).sort_values(ascending=False)
print("TOP 10 Elo entering the 2026 World Cup:")
print(ranking.head(10).round(0).to_string())

TOP 10 Elo entering the 2026 World Cup:
Argentina      1741.0
Netherlands    1723.0
Germany        1720.0
France         1711.0
Brazil         1700.0
Croatia        1656.0
Belgium        1619.0
Italy          1617.0
England        1592.0
Sweden         1577.0


**Sanity check:** Argentina (2022 champions) lead, with France, Netherlands, Germany and Croatia
up top — the ranking passes the smell test. Caveat: ratings only update in World Cup matches, so
teams absent from recent editions (Italy, Sweden) keep stale, inflated ratings. A production Elo
would also use qualifiers and friendlies.

## 2. Does Elo beat the v0 features? Building the v1 dataset

In [3]:
home = matches_m[["year", "match_id", "home_team_name", "away_team_name",
                  "home_team_score", "away_team_score", "home_team_win", "draw"]].copy()
home.columns = ["year", "match_id", "team", "opponent", "goals_for", "goals_against", "won", "draw"]
away = matches_m[["year", "match_id", "away_team_name", "home_team_name",
                  "away_team_score", "home_team_score", "away_team_win", "draw"]].copy()
away.columns = ["year", "match_id", "team", "opponent", "goals_for", "goals_against", "won", "draw"]
team_matches = pd.concat([home, away], ignore_index=True)

def team_features(team, year, prefix):
    past = team_matches[(team_matches["team"] == team) & (team_matches["year"] < year)]
    if len(past) == 0:
        return {f"{prefix}_wc_played": 0, f"{prefix}_win_rate": 0.0, f"{prefix}_goals_for": 0.0,
                f"{prefix}_goals_against": 0.0, f"{prefix}_recent_form": 0.0}
    last_two = sorted(past["year"].unique())[-2:]
    recent = past[past["year"].isin(last_two)]
    return {
        f"{prefix}_wc_played": past["year"].nunique(),
        f"{prefix}_win_rate": past["won"].mean(),
        f"{prefix}_goals_for": past["goals_for"].mean(),
        f"{prefix}_goals_against": past["goals_against"].mean(),
        f"{prefix}_recent_form": recent["won"].mean(),
    }

In [4]:
tournaments = pd.read_csv(PATH + "/historical/tournaments.csv")
tournaments_m = tournaments[tournaments["tournament_name"].str.contains("Men's")].copy()
tournaments_m["year"] = tournaments_m["tournament_name"].str[:4].astype(int)
host_by_year = tournaments_m.set_index("year")["host_country"].to_dict()

train_matches = matches_m[(matches_m["year"] >= 1962) & (matches_m["group_stage"] == 1)].copy()

rows = []
for _, m in train_matches.iterrows():
    row = {"year": m["year"]}
    row.update(team_features(m["home_team_name"], m["year"], "home"))
    row.update(team_features(m["away_team_name"], m["year"], "away"))
    host = host_by_year.get(m["year"], "")
    row["home_is_host"] = int(m["home_team_name"] == host)
    row["away_is_host"] = int(m["away_team_name"] == host)
    # NEW: pre-match Elo (already computed, leakage-free by construction)
    row["elo_home"] = m["elo_home"]
    row["elo_away"] = m["elo_away"]
    row["elo_diff"] = m["elo_diff"]
    row["target"] = 0 if m["home_team_win"] == 1 else (1 if m["draw"] == 1 else 2)
    rows.append(row)

dataset = pd.DataFrame(rows)
dataset["diff_win_rate"] = dataset["home_win_rate"] - dataset["away_win_rate"]
dataset["diff_form"] = dataset["home_recent_form"] - dataset["away_recent_form"]
print(dataset.shape)

(636, 19)


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

FEATURES_V0 = ["home_wc_played", "home_win_rate", "home_goals_for", "home_goals_against",
               "home_recent_form", "home_is_host", "away_wc_played", "away_win_rate",
               "away_goals_for", "away_goals_against", "away_recent_form", "away_is_host",
               "diff_win_rate", "diff_form"]
FEATURES_V1 = FEATURES_V0 + ["elo_home", "elo_away", "elo_diff"]

train = dataset[dataset["year"] <= 2014]
test  = dataset[dataset["year"] >= 2018]

for name, feats in [("v0 (sin Elo)", FEATURES_V0), ("v1 (con Elo)", FEATURES_V1)]:
    rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)
    rf.fit(train[feats], train["target"])
    acc = accuracy_score(test["target"], rf.predict(test[feats]))
    print(f"{name}: {acc*100:.1f}%")

v0 (sin Elo): 49.0%
v1 (con Elo): 49.0%


In [6]:
rf_v1 = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)
rf_v1.fit(train[FEATURES_V1], train["target"])

importances = pd.Series(rf_v1.feature_importances_, index=FEATURES_V1).sort_values(ascending=False)
print(importances.round(3).to_string())

away_wc_played        0.113
home_goals_for        0.108
elo_diff              0.089
diff_win_rate         0.081
elo_home              0.073
elo_away              0.072
diff_form             0.071
home_goals_against    0.058
home_win_rate         0.054
away_goals_for        0.053
away_goals_against    0.050
away_recent_form      0.047
home_recent_form      0.043
home_wc_played        0.040
away_win_rate         0.040
home_is_host          0.004
away_is_host          0.002


In [15]:
intl = pd.read_csv(PATH + "/historical/all_international_results.csv")
intl["date"] = pd.to_datetime(intl["date"])

# Played matches only (the file even includes scheduled 2026 fixtures!)
intl = intl[intl["home_score"].notna()].copy()

# Harmonize names with our datasets
intl["home_team"] = intl["home_team"].replace(HEIR_NATIONS)
intl["away_team"] = intl["away_team"].replace(HEIR_NATIONS)

# East Germany appears as "German DR" in this dataset
intl["home_team"] = intl["home_team"].replace({"German DR": "East Germany"})
intl["away_team"] = intl["away_team"].replace({"German DR": "East Germany"})

intl = intl.sort_values("date").reset_index(drop=True)
print(len(intl), "played matches |", intl["date"].min().date(), "->", intl["date"].max().date())
print(intl["tournament"].value_counts().head(8))

49405 played matches | 1872-11-30 -> 2026-06-10
tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                            964
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
Name: count, dtype: int64


In [16]:
def k_factor(tournament):
    if "FIFA World Cup" in tournament and "qualification" not in tournament.lower():
        return 60
    if any(t in tournament for t in ["UEFA Euro", "Copa América", "African Cup", "AFC Asian Cup", "Gold Cup"]):
        return 50
    if "qualification" in tournament.lower() or "Nations League" in tournament:
        return 40
    return 20  # friendlies and minor tournaments

elo2 = {}
pre_home, pre_away = [], []

for row in intl.itertuples(index=False):
    h, a = row.home_team, row.away_team
    r_h = elo2.get(h, 1500)
    r_a = elo2.get(a, 1500)
    pre_home.append(r_h)
    pre_away.append(r_a)

    if row.home_score > row.away_score:
        score = 1.0
    elif row.home_score == row.away_score:
        score = 0.5
    else:
        score = 0.0

    K = k_factor(row.tournament)
    exp = expected_score(r_h, r_a)
    elo2[h] = r_h + K * (score - exp)
    elo2[a] = r_a + K * ((1 - score) - (1 - exp))

intl["elo_home_pre"] = pre_home
intl["elo_away_pre"] = pre_away

ranking2 = pd.Series(elo2).sort_values(ascending=False)
print("TOP 15 Elo TODAY (entering the 2026 World Cup):")
print(ranking2.head(15).round(0).to_string())

TOP 15 Elo TODAY (entering the 2026 World Cup):
Spain          2087.0
Argentina      2054.0
France         2007.0
England        1974.0
Portugal       1952.0
Brazil         1944.0
Germany        1942.0
Colombia       1930.0
Morocco        1923.0
Netherlands    1923.0
Ecuador        1914.0
Japan          1907.0
Mexico         1905.0
Turkey         1894.0
Croatia        1882.0


## 4. Plugging the new Elo into the training data

In [17]:
elo_cols = intl[["date", "home_team", "away_team", "elo_home_pre", "elo_away_pre"]]

merged = matches_m.merge(
    elo_cols,
    left_on=["match_date", "home_team_name", "away_team_name"],
    right_on=["date", "home_team", "away_team"],
    how="left",
)

# Coverage check: how many WC matches found their Elo?
missing = merged["elo_home_pre"].isnull().sum()
print(f"WC matches: {len(merged)} | without Elo match: {missing} ({missing/len(merged)*100:.1f}%)")

# If any are missing, see who they are (likely name mismatches)
if missing > 0:
    print(merged[merged["elo_home_pre"].isnull()][["match_date", "home_team_name", "away_team_name"]].head(15).to_string())

WC matches: 964 | without Elo match: 139 (14.4%)
   match_date home_team_name away_team_name
1  1930-07-13  United States        Belgium
2  1930-07-14         Serbia         Brazil
3  1930-07-14        Romania           Peru
6  1930-07-17         Serbia        Bolivia
7  1930-07-17  United States       Paraguay
11 1930-07-20         Brazil        Bolivia
12 1930-07-20       Paraguay        Belgium
20 1934-05-27        Germany        Belgium
21 1934-05-27        Hungary          Egypt
23 1934-05-27          Spain         Brazil
24 1934-05-27         Sweden      Argentina
25 1934-05-27    Switzerland    Netherlands
33 1934-06-07        Germany        Austria
35 1938-06-04    Switzerland        Germany
43 1938-06-09    Switzerland        Germany


In [18]:
missing_matches = merged[merged["elo_home_pre"].isnull()]
print(missing_matches["year"].value_counts().sort_index().to_string())
print(f"\nMissing in our training window (>=1962): {(missing_matches['year'] >= 1962).sum()}")

year
1930     7
1934     6
1938     6
1950    14
1954    12
1958    13
1962    14
1966    13
1970    14
1974     5
1978    12
1982     2
1986     4
1990     4
1994     1
1998     2
2002     4
2006     1
2010     1
2014     1
2018     2
2022     1

Missing in our training window (>=1962): 81


In [19]:
# Second attempt for the missing ones: swapped home/away order
swapped = matches_m.merge(
    elo_cols,
    left_on=["match_date", "home_team_name", "away_team_name"],
    right_on=["date", "away_team", "home_team"],  # ← swapped!
    how="left",
)
# Careful: if matched swapped, the Elo columns are also swapped
merged["elo_home_pre"] = merged["elo_home_pre"].fillna(swapped["elo_away_pre"])
merged["elo_away_pre"] = merged["elo_away_pre"].fillna(swapped["elo_home_pre"])

missing2 = merged["elo_home_pre"].isnull().sum()
print(f"After swapped merge: {missing2} missing ({missing2/len(merged)*100:.1f}%)")
print("Missing >=1962:", (merged[merged['elo_home_pre'].isnull()]['year'] >= 1962).sum())

After swapped merge: 0 missing (0.0%)
Missing >=1962: 0


In [21]:
train_matches2 = merged[(merged["year"] >= 1962) & (merged["group_stage"] == 1) &
                        (merged["elo_home_pre"].notna())].copy()

rows = []
for _, m in train_matches2.iterrows():
    row = {"year": m["year"]}
    row.update(team_features(m["home_team_name"], m["year"], "home"))
    row.update(team_features(m["away_team_name"], m["year"], "away"))
    host = host_by_year.get(m["year"], "")
    row["home_is_host"] = int(m["home_team_name"] == host)
    row["away_is_host"] = int(m["away_team_name"] == host)
    # The NEW Elo: fed with ALL international matches (form included!)
    row["elo_home"] = m["elo_home_pre"]
    row["elo_away"] = m["elo_away_pre"]
    row["elo_diff"] = m["elo_home_pre"] - m["elo_away_pre"]
    row["target"] = 0 if m["home_team_win"] == 1 else (1 if m["draw"] == 1 else 2)
    rows.append(row)

dataset2 = pd.DataFrame(rows)
dataset2["diff_win_rate"] = dataset2["home_win_rate"] - dataset2["away_win_rate"]
dataset2["diff_form"] = dataset2["home_recent_form"] - dataset2["away_recent_form"]

train2 = dataset2[dataset2["year"] <= 2014]
test2  = dataset2[dataset2["year"] >= 2018]

for name, feats in [("v0 (sin Elo)", FEATURES_V0), ("v1 (Elo internacional)", FEATURES_V1),
                    ("Solo Elo + host", ["elo_home", "elo_away", "elo_diff", "home_is_host", "away_is_host"])]:
    rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)
    rf.fit(train2[feats], train2["target"])
    acc = accuracy_score(test2["target"], rf.predict(test2[feats]))
    print(f"{name}: {acc*100:.1f}%")

v0 (sin Elo): 49.0%
v1 (Elo internacional): 50.0%
Solo Elo + host: 47.9%


In [22]:
# Robust comparison: test on each World Cup separately (training on everything before it)
results = []
for test_year in [1998, 2002, 2006, 2010, 2014, 2018, 2022]:
    tr = dataset2[dataset2["year"] < test_year]
    te = dataset2[dataset2["year"] == test_year]
    row = {"test_year": test_year, "n": len(te)}
    for name, feats in [("v0", FEATURES_V0), ("v1", FEATURES_V1)]:
        rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)
        rf.fit(tr[feats], tr["target"])
        row[name] = round(accuracy_score(te["target"], rf.predict(te[feats])) * 100, 1)
    results.append(row)

comparison = pd.DataFrame(results)
comparison["winner"] = np.where(comparison["v1"] > comparison["v0"], "v1",
                       np.where(comparison["v1"] < comparison["v0"], "v0", "tie"))
print(comparison.to_string(index=False))
print(f"\nMedia v0: {comparison['v0'].mean():.1f}% | Media v1: {comparison['v1'].mean():.1f}%")

 test_year  n   v0   v1 winner
      1998 48 54.2 58.3     v1
      2002 48 41.7 41.7    tie
      2006 48 47.9 52.1     v1
      2010 48 45.8 56.2     v1
      2014 48 50.0 54.2     v1
      2018 48 52.1 56.2     v1
      2022 48 47.9 47.9    tie

Media v0: 48.5% | Media v1: 52.4%


**Robust validation (leave-one-tournament-out):** v1 wins in 5 of 7 World Cups and never loses.
Mean accuracy: 52.4% vs 48.5% (+3.9 pts). The single 2018-2022 test understated the gain —
small test sets mislead; cross-validation reveals the truth. **v1 is the new production model.**

## 6. 2026 predictions with model v1

In [23]:
# Final model: train on ALL available history (1962-2022)
rf_v1 = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)
rf_v1.fit(dataset2[FEATURES_V1], dataset2["target"])

HOSTS_2026 = {"United States", "Mexico", "Canada"}
groups_2026 = fixture[fixture["group"].notna()].copy()

rows = []
for _, m in groups_2026.iterrows():
    row = {"round": m["round"], "date": m["date"], "time": m["time"], "group": m["group"],
           "home_team": m["team1"], "away_team": m["team2"]}
    row.update(team_features(m["team1"], 2026, "home"))
    row.update(team_features(m["team2"], 2026, "away"))
    row["home_is_host"] = int(m["team1"] in HOSTS_2026)
    row["away_is_host"] = int(m["team2"] in HOSTS_2026)
    # Current Elo (after ALL internationals up to June 2026)
    row["elo_home"] = elo2.get(m["team1"], 1500)
    row["elo_away"] = elo2.get(m["team2"], 1500)
    row["elo_diff"] = row["elo_home"] - row["elo_away"]
    rows.append(row)

pred26 = pd.DataFrame(rows)
pred26["diff_win_rate"] = pred26["home_win_rate"] - pred26["away_win_rate"]
pred26["diff_form"] = pred26["home_recent_form"] - pred26["away_recent_form"]

probs = rf_v1.predict_proba(pred26[FEATURES_V1])
pred26["p_home_win"] = (probs[:, 0] * 100).round(1)
pred26["p_draw"]     = (probs[:, 1] * 100).round(1)
pred26["p_away_win"] = (probs[:, 2] * 100).round(1)
pred26["prediction"] = np.array(["Home win", "Draw", "Away win"])[probs.argmax(axis=1)]

# Spain kickoff times (same as v0)
import re
def to_spain_time(date_str, time_str):
    if pd.isna(time_str):
        return None
    mt = re.match(r"(\d{2}):(\d{2}) UTC([+-]\d+)", time_str)
    if not mt:
        return None
    h, mi, off = int(mt.group(1)), int(mt.group(2)), int(mt.group(3))
    return pd.Timestamp(f"{date_str} {h:02d}:{mi:02d}") - pd.Timedelta(hours=off) + pd.Timedelta(hours=2)

pred26["kickoff_spain"] = [to_spain_time(d, t) for d, t in zip(pred26["date"], pred26["time"])]

# Save as v1 — the v0 file stays untouched (matchday 1 was published with it)
pred26.to_csv("../data/processed/predictions_2026_group_stage_v1.csv", index=False)
print("Saved v1 predictions!")

# Curiosity: where do v0 and v1 disagree the most?
v0 = pd.read_csv("../data/processed/predictions_2026_group_stage_v0.csv")
both = pred26.merge(v0[["home_team", "away_team", "prediction"]], on=["home_team", "away_team"], suffixes=("_v1", "_v0"))
disagreements = both[both["prediction_v1"] != both["prediction_v0"]]
print(f"\nv0 y v1 discrepan en {len(disagreements)} de 72 partidos:")
print(disagreements[["home_team", "away_team", "prediction_v0", "prediction_v1"]].to_string(index=False))

Saved v1 predictions!

v0 y v1 discrepan en 16 de 72 partidos:
  home_team    away_team prediction_v0 prediction_v1
     Mexico  South Korea      Away win          Draw
   Scotland      Morocco      Home win      Away win
    Morocco        Haiti      Home win          Draw
  Australia       Turkey      Home win      Away win
Ivory Coast      Ecuador      Home win      Away win
    Curaçao  Ivory Coast      Home win      Away win
Netherlands       Sweden          Draw      Home win
    Tunisia        Japan      Home win      Away win
New Zealand        Egypt      Home win      Away win
      Egypt         Iran      Home win      Away win
 Cape Verde Saudi Arabia      Home win      Away win
       Iraq       Norway      Home win      Away win
     Norway      Senegal      Home win      Away win
   Colombia     Portugal      Home win      Away win
    England      Croatia      Home win      Away win
      Ghana       Panama          Draw      Away win


In [24]:
weird = pred26[(pred26["home_team"] == "Morocco") & (pred26["away_team"] == "Haiti")]
print(weird[["home_team", "away_team", "elo_home", "elo_away", "p_home_win", "p_draw", "p_away_win"]].to_string())

# ¿Y cómo trata a los debutantes en general? (equipos sin historia mundialista)
debutants = pred26[pred26["away_wc_played"] == 0]
print(debutants[["home_team", "away_team", "p_home_win", "p_draw", "p_away_win", "prediction"]].to_string(index=False))

   home_team away_team     elo_home     elo_away  p_home_win  p_draw  p_away_win
17   Morocco     Haiti  1923.188273  1628.454765        40.2    42.0        17.7
home_team  away_team  p_home_win  p_draw  p_away_win prediction
  Germany    Curaçao        73.2    18.7         8.1   Home win
  Ecuador    Curaçao        50.8    31.0        18.2   Home win
    Spain Cape Verde        71.1    16.4        12.5   Home win
  Uruguay Cape Verde        76.0    15.3         8.7   Home win
  Austria     Jordan        67.6    18.3        14.1   Home win
 Portugal Uzbekistan        74.8    14.9        10.3   Home win
 DR Congo Uzbekistan        29.9    24.8        45.3   Away win


**Debutant check:** teams with no WC history get sensible predictions (their Elo from
qualifiers/friendlies fills the gap). Morocco-Haiti (42% draw vs 40% home) is a marginal
call, not a bug — and a reminder that the pick is just the argmax of honest probabilities.

In [25]:
# Save the internationals + pre-match Elo as an artifact for other notebooks
intl.to_csv("../data/processed/internationals_with_elo.csv", index=False)
print("Artifact saved!")

Artifact saved!
